##  1. 데이터 준비

### 1) 네이버 경제 뉴스

In [1]:
import pandas as pd

df_path_001 = r'./data/naver_economic_news.csv'

df_data_001 = pd.read_csv(df_path_001)
df_data_001.head()

,일자,섹션,제목,언론사,본문
0,2026-01-01,258,"잘 나가는 방위산업株…한화에어로·LIG넥스원, 나란히 AA로 신용 ‘레벨업’ [투자...",헤럴드경제,"2025년 활약한 방위산업株, 신용평가 등급 A+~AA 포진 한화에어로·LIG넥스원..."
1,2026-01-01,258,"베인캐피털, 안다르 모회사 에코마케팅 잔여지분 공개 매수",매일경제,2800억들여 이달 21일까지 완료 후 상장폐지 추진 목표 글로벌 사모펀드(PEF)...
2,2026-01-01,258,"[마켓인]“불완전 증자”vs“이미 주주 등재”…고려아연 유증, 등기 지연 논란",이데일리,지난 26일 대금 납입 마쳤지만 서울지법 등기수리 통상 일정대비 지연 고려아연 “이...
3,2026-01-01,258,고려아연 유상증자 공방…신주 등기 막판 변수로 [시그널],서울경제,중앙지법 등기국서 아직 수리 안 해 美 JV 주총서 의결권 행사 못할수도 정정 공시...
4,2026-01-01,258,"고려아연 ""美 합작법인 신주 발행 통상적 진행중…등기 불발 등 사실 아냐""",서울경제,"""이사회, 美 달러화 기준 발행가액 및 총액 의결 대금 납입도 완료···환전 없이 ..."


### 2) 코스피 지수

In [2]:
df_path_002 = r'./data/KOSPI_raw_data.csv'

df_data_002 = pd.read_csv(df_path_002)
df_data_002.head()

,Price,Close,High,Low,Open,Volume
0,Ticker,^KS11,^KS11,^KS11,^KS11,^KS11
1,Date,NaN,NaN,NaN,NaN,NaN
2,2025-12-22,4105.93017578125,4105.93017578125,4083.1298828125,4096.259765625,315600
3,2025-12-23,4117.31982421875,4140.83984375,4110.25,4127.39990234375,385800
4,2025-12-24,4108.6201171875,4137.2001953125,4106.6201171875,4136.240234375,363600


In [3]:
#   하나의 행: Series 객체
#   열 이름: idx로 개별 변경 불가능
#   list 형태로 반환하여 변경 후, 통쨰로 변경必
df_data_002_col = df_data_002.columns.tolist()
df_data_002_col[0] = 'Date'
df_data_002.columns = df_data_002_col
df_data_002.columns

#   1~2행 제거
df_data_002 = df_data_002.drop([0, 1], axis=0)
df_data_002.head()

,Date,Close,High,Low,Open,Volume
2,2025-12-22,4105.93017578125,4105.93017578125,4083.1298828125,4096.259765625,315600
3,2025-12-23,4117.31982421875,4140.83984375,4110.25,4127.39990234375,385800
4,2025-12-24,4108.6201171875,4137.2001953125,4106.6201171875,4136.240234375,363600
5,2025-12-26,4129.68017578125,4143.14013671875,4116.52978515625,4130.3701171875,510200
6,2025-12-29,4220.56005859375,4220.56005859375,4146.47998046875,4146.47998046875,502400


In [4]:
print(df_data_002.info())

#   데이터 속성 변경
#   errors = 'coerce': 에러가 출력되면 무시+강제로 결측 처리
df_data_002['Close'] = pd.to_numeric(df_data_002['Close'], errors = 'coerce')
df_data_002['High'] = pd.to_numeric(df_data_002['High'], errors = 'coerce')
df_data_002['Low'] = pd.to_numeric(df_data_002['Low'], errors = 'coerce')
df_data_002['Open'] = pd.to_numeric(df_data_002['Open'], errors = 'coerce')
df_data_002['Volume'] = pd.to_numeric(df_data_002['Volume'], errors = 'coerce')
print(df_data_002.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 2 to 99
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    98 non-null     object
 1   Close   98 non-null     object
 2   High    98 non-null     object
 3   Low     98 non-null     object
 4   Open    98 non-null     object
 5   Volume  98 non-null     object
dtypes: object(6)
memory usage: 4.7+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98 entries, 2 to 99
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    98 non-null     object 
 1   Close   98 non-null     float64
 2   High    98 non-null     float64
 3   Low     98 non-null     float64
 4   Open    98 non-null     float64
 5   Volume  98 non-null     int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 4.7+ KB
None


In [5]:
#   종가(Close) 열 5이평선 생성
df_data_002['MA5'] = df_data_002['Close'].rolling(window=5).mean()

#   5거래일 이후 이평선 현재 행에 위치
df_data_002['pro_MA5'] = df_data_002['MA5'].shift(-5)

#   answer 열 생성
#   5거래일 이후 및 현재 이평선 비교
df_data_002['answer'] = ((df_data_002['pro_MA5'] / df_data_002['MA5']) - 1) >= 0.03

#   'answer' 열 빈도 확인
df_data_002['answer'].value_counts()

answer
True     58
False    40
Name: count, dtype: int64

### 3) 데이터 병합

In [6]:
#   각 데이터 프레임의 '일자' 열 및 'Date' 열에 대해 속성 변경
df_data_001['일자'] = pd.to_datetime(df_data_001['일자'])
df_data_002['Date'] = pd.to_datetime(df_data_002['Date'])

#   실제 개장일 날짜 추출:  고유값 출력하여 df 생성 후, 정렬
trading_date = pd.DataFrame({'Target_Date': df_data_002['Date'].unique()})
trading_date = trading_date.sort_values('Target_Date')
df_data_001 = df_data_001.sort_values('일자')

#   merge_asof: 최근접 값 기준으로 병합(공휴일 및 주말 처리)
#   direction='forward': 현자 날짜보다 크거나 같은 날 중 근접 값
df_news = pd.merge_asof(
    df_data_001, trading_date, left_on = '일자', right_on = 'Target_Date', direction = 'forward'
)

df_news.head()

,일자,섹션,제목,언론사,본문,Target_Date
0,2026-01-01,258,"잘 나가는 방위산업株…한화에어로·LIG넥스원, 나란히 AA로 신용 ‘레벨업’ [투자...",헤럴드경제,"2025년 활약한 방위산업株, 신용평가 등급 A+~AA 포진 한화에어로·LIG넥스원...",2026-01-02
1,2026-01-01,263,오세훈 서울시장 “비상계엄 등 잘못 인정하고 반성해야”,이코노미스트,국민의힘 계엄 반성 등 과거와의 단절 요구 ‘국민이 먹고 사는 문제’ 집중해야 한다...,2026-01-02
2,2026-01-01,263,전기차 국고보조금 작년과 동일…내연차 폐차·매각후 전기차 사면 100만원 더,부산일보,"기후부, 전기차 구매보조금 개편안 발표 5300만원 미만 차 보조금 100% 지급 ...",2026-01-02
3,2026-01-01,263,새해 첫날 일부 복권판매점서 '로또발행 일시 중단' 발생,국제신문,1일 오전 서울 등 일부 판매점서 발행 중단 복권 운영사 동행복권에 약 50건 민원...,2026-01-02
4,2026-01-01,263,"배경훈 ""쿠팡, 5개월치 홈피 접속로그 삭제 방치…법 위반""",연합뉴스TV,'쿠팡 사태 범정부 TF' 팀장인 배경훈 부총리 겸 과기정통부 장관은 쿠팡 측의 과...,2026-01-02


In [7]:
#  df_data_002에서 필요한 열만 추출 후 df_target 생성
df_stock = df_data_002[['Date', 'answer']]
                         
#   df_data_001 '일자' 기준 병합
df_merge = pd.merge(df_news, df_stock, left_on='Target_Date', right_on='Date', how='left')
df_merge.head()

#   중복 및 불필요한 열 제거
df_merge = df_merge.drop(['일자', 'Date'], axis = 1)
df_merge.columns

Index(['섹션', '제목', '언론사', '본문', 'Target_Date', 'answer'], dtype='object')

## 2. 데이터 전처리

### 1) 데이터 결측값 확인

In [8]:
#   df_merge 열별 결측값 확인
for col in df_merge.columns.tolist():
    print(df_merge.isnull()[col].value_counts())

#   '본문' 열 결측 처리
df_merge.dropna(subset = ['본문'], inplace=True)

#   '본문' 열 결측 처리 확인
df_merge['본문'].isnull().value_counts(dropna=False)

섹션
False    693926
Name: count, dtype: int64
제목
False    693926
Name: count, dtype: int64
언론사
False    693926
Name: count, dtype: int64
본문
False    689455
True       4471
Name: count, dtype: int64
Target_Date
False    693926
Name: count, dtype: int64
answer
False    693926
Name: count, dtype: int64


본문
False    689455
Name: count, dtype: int64

### 2) 데이터 중복값 확인 및 제거

In [9]:
#   df_merge 열별 중복값 확인
for col in df_merge.columns.tolist():
    print(df_merge[col].duplicated().value_counts())

#   '제목' 열 기준 중복값 처리
df_merge.drop_duplicates(subset=['제목'], inplace=True)

df_merge['제목'].duplicated().value_counts()

섹션
True     689447
False         8
Name: count, dtype: int64
제목
False    539221
True     150234
Name: count, dtype: int64
언론사
True     689374
False        81
Name: count, dtype: int64
본문
False    544359
True     145096
Name: count, dtype: int64
Target_Date
True     689374
False        81
Name: count, dtype: int64
answer
True     689453
False         2
Name: count, dtype: int64


제목
False    539221
Name: count, dtype: int64

### 3) 텍스트 정제

In [10]:
import re

#   '수정'열: 한글, 영문(대소문자), 숫자, 공백 제외 문자 제거
df_merge['수정'] = df_merge['본문'].apply(lambda x: re.sub('[^ A-Za-z0-9ㄱ-ㅣ가-힣]', ' ', x))

#   '수정'열: 문장의 시작 공백을 ""으로 치환 
df_merge['수정'] = df_merge['수정'].apply(lambda x: re.sub('^ +', '', x))

#   '수정'열: 공란을 결측치로 변환
df_merge['수정'] = df_merge['수정'].replace('', None)

#   '수정'열 결측 여부 확인
print(df_merge['수정'].isnull().value_counts(dropna=False))

#   '수정'열 결측 제거
df_merge = df_merge.dropna(subset='수정')

#   '수정'열 결측 제거 확인
print(df_merge['수정'].isnull().value_counts(dropna=False))

수정
False    539220
True          1
Name: count, dtype: int64
수정
False    539220
Name: count, dtype: int64


In [11]:
#   '수정'열 중복 확인
print(df_merge['수정'].duplicated().sum())

#   '수정'열 중복 제거
df_merge.drop_duplicates(subset=['수정'], inplace=True)

#   '수정'열 중복 제거 확인
print(df_merge['수정'].duplicated().sum())

14414
0


###  4) 토큰화

In [12]:
#   kiwipiepy 활용, kiwi_tokenizer 생성
from kiwipiepy import Kiwi

kiwi = Kiwi()

#   대상품사 설정: 일반명사, 고유명사, 동사, 형용사
stnd_poses = ['NNG', 'NNP', 'VV', 'VA']
#   금칙어 설정
stop_words = []

def kiwi_tokenizer(target_article):
    text = str(target_article)
    #   Kiwi 형태소 분석기 생성
    tokens = kiwi.tokenize(text)
    token_list = [token.form for token in tokens
        #   품사 설정
        if token.tag in stnd_poses and
        #   불용어 제외
        token.form not in stop_words
        ]
    return token_list

from tqdm import tqdm
tqdm.pandas()

df_merge['tokens'] = df_merge['수정'].progress_apply(kiwi_tokenizer)  

  0%|          | 2403/524806 [00:24<1:29:47, 96.97it/s] 


KeyboardInterrupt: 

In [ ]:
#   토큰화 결과 저장
df_merge.to_pickle('vectorizer_file.pkl')